# 01c — Classical Features for Dataset 3 (Intercounty Migration)

Replicates the classical-feature extraction from `01_data_exploration.ipynb` for **Dataset 3** —
a static directed weighted network from the UF Sparse Matrix Collection
(`HB/psmigr_1`, Paul Slater 1983, intercounty migration in the US).

Unlike Datasets 1 and 2, this graph has **no Equity/Assets information and no targets**,
so DebtRank is not computed. We compute the six purely topological features
(degree centrality, weighted degree, betweenness centrality, closeness centrality,
eigenvector centrality, PageRank) plus basic network summary statistics.

Input:  `src/datasets/dataset_3/econ-psmigr1.mtx` (3140 nodes, 543162 directed edges)

Output: `src/data/classical_features_dataset3/classical_features_dataset3.parquet`

In [ ]:
import sys
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import networkx as nx
from scipy.io import mmread

from src.models import (
    degree_centrality,
    betweenness_centrality,
    closeness_centrality,
    eigenvector_centrality,
    weighted_degree,
    pagerank_centrality,
)

DATASET3_PATH = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_3' / 'econ-psmigr1.mtx'
OUT_FEATURES = PROJECT_ROOT / 'src' / 'data' / 'classical_features_dataset3'
OUT_FEATURES.mkdir(parents=True, exist_ok=True)

print(f'Dataset 3 file : {DATASET3_PATH}')
print(f'Output dir     : {OUT_FEATURES}')

## Load Dataset 3

Read the Matrix Market file and convert it into the `edges` / `nodes` DataFrame schema
the centrality functions in `src.models` expect (`Sourceid`, `Targetid`, `Weights` for edges,
`index` for nodes). Node IDs in the `.mtx` file are 1-indexed; we convert to 0-indexed
to stay consistent with the rest of the project.

In [ ]:
sparse_matrix = mmread(DATASET3_PATH).tocoo()
n_nodes = sparse_matrix.shape[0]

edges = pd.DataFrame({
    'Sourceid': sparse_matrix.row.astype(int),
    'Targetid': sparse_matrix.col.astype(int),
    'Weights':  sparse_matrix.data.astype(float),
})
edges = edges[edges['Weights'] > 0].reset_index(drop=True)

nodes = pd.DataFrame({'index': np.arange(n_nodes, dtype=int)})

print(f'Nodes: {len(nodes)}')
print(f'Edges: {len(edges)}')
display(edges.head())

## Network Summary Statistics

Topological diagnostics: density, components, isolated nodes, edge-weight distribution.

In [ ]:
G = nx.from_pandas_edgelist(
    edges, 'Sourceid', 'Targetid', 'Weights', create_using=nx.DiGraph()
)
G.add_nodes_from(nodes['index'].tolist())

weakly_components = list(nx.weakly_connected_components(G))
strongly_components = list(nx.strongly_connected_components(G))
isolated = [n for n in G.nodes if G.in_degree(n) == 0 and G.out_degree(n) == 0]
self_loops = list(nx.selfloop_edges(G))

summary = {
    'nodes': G.number_of_nodes(),
    'edges': G.number_of_edges(),
    'self_loops': len(self_loops),
    'density': nx.density(G),
    'reciprocity': nx.reciprocity(G),
    'weakly_connected_components': len(weakly_components),
    'largest_wcc_size': max((len(c) for c in weakly_components), default=0),
    'strongly_connected_components': len(strongly_components),
    'largest_scc_size': max((len(c) for c in strongly_components), default=0),
    'isolated_nodes': len(isolated),
    'min_edge_weight': float(edges['Weights'].min()),
    'median_edge_weight': float(edges['Weights'].median()),
    'max_edge_weight': float(edges['Weights'].max()),
    'total_edge_weight': float(edges['Weights'].sum()),
}

summary_df = pd.DataFrame(
    [(k, v) for k, v in summary.items()], columns=['statistic', 'value']
)
display(summary_df)

## Compute Classical Features

Six features: degree, weighted degree, betweenness, closeness, PageRank, eigenvector.

**Note**: betweenness centrality on a directed weighted graph of this size (3140 nodes,
~540k edges) is the slowest step and can take several minutes.

In [ ]:
deg_df  = degree_centrality(edges, nodes)
wdeg_df = weighted_degree(edges, nodes)
pr_df   = pagerank_centrality(edges, nodes)
clo_df  = closeness_centrality(edges, nodes)
bet_df  = betweenness_centrality(edges, nodes)

features = nodes.copy().rename(columns={'index': 'node_id'})
for df in (deg_df, wdeg_df, pr_df, clo_df, bet_df):
    features = features.merge(
        df.rename(columns={'bank_id': 'node_id'}), on='node_id', how='left'
    )

out_path = OUT_FEATURES / 'classical_features_dataset3.parquet'
features.to_parquet(out_path, index=False)
print(f'[OK] saved features  shape={features.shape}  -> {out_path}')

## Add Eigenvector Centrality

Computed separately — may fail to converge on disconnected or near-degenerate components
and is patched into the saved parquet.

In [ ]:
eig_df = eigenvector_centrality(edges, nodes)

out_path = OUT_FEATURES / 'classical_features_dataset3.parquet'
features = pd.read_parquet(out_path)
features = features.merge(
    eig_df.rename(columns={'bank_id': 'node_id'}), on='node_id', how='left'
)
features.to_parquet(out_path, index=False)
print(f'[OK] added eigenvector_centrality  shape={features.shape}')

## Verify Output

In [ ]:
df = pd.read_parquet(OUT_FEATURES / 'classical_features_dataset3.parquet')
print(f'Shape: {df.shape}')
display(df.head())

feature_cols = [
    'degree_centrality_total', 'weighted_degree_total',
    'betweenness_centrality', 'closeness_centrality',
    'eigenvector_centrality', 'pagerank',
]
print('\nFeature summary:')
display(df[feature_cols].describe())